# Phase 0 — what an epoch costs

The first experiment in `docs/training-plan.md`. It decides three things the rest of the
plan is built on: the **batch size**, the **dataloader width**, and whether **grouped
cross-validation** is affordable.

**This has been run.** Measured on a Colab T4, 8 vCPUs, 2026-09-07. Results are in
`output/phase0/`. Re-run it when the hardware changes — the numbers below are properties of
that machine, not of the code.

| model | params | epoch | peak mem | GPU util |
|---|---|---|---|---|
| `baseline_cnn` 64x64 | 157k | 23.6s (12.8s at 8 workers) | 0.29 GiB | 15.7% |
| `dilated_style` 64x64 | 298k | 168.4s | 3.07 GiB | 97.4% |
| `densenet_style` 64x64 | 304k | 163.6s | 3.94 GiB | 96.3% |
| `vit_b_16` frozen 224 | 86M | 766s | 1.35 GiB | 98.1% |
| `vit_b_16` fine-tune 224 | 86M | 1482s | 9.80 GiB | 98.9% |
| `vit_b_32` fine-tune 224 | 86M | 458s | 3.47 GiB | 95.9% |

**What it changed:**

1. **`num_workers: 2` -> `8`.** The VM has 8 cores. This alone was a **1.9x** speedup on the
   small models (6,258 -> 12,852 samples/s) and costs nothing.
2. **`batch_size: 512` -> `256`.** Throughput is flat across 64-1024 on the 64x64 models
   (5,642 -> 6,403 samples/s for a 16x batch increase), so a large batch buys no speed and
   costs optimizer steps: batch 512 gives 4,740 updates in a 20-epoch run, batch 256 gives
   9,460.
3. **`vit_b_16` batch 32 -> 128.** The config claimed 32 was "what fits a T4". 128 fits in
   9.8 GiB and is 7% faster.

**The thing that surprised us:** parameter count predicts nothing. `dilated_style` and
`densenet_style` carry ~2x `baseline_cnn`'s parameters and cost **7x** the time and **10x**
the memory, because they hold full 64x64 resolution deep into the network.

## 1. Environment

In [ ]:
import os, platform, subprocess, sys

import torch

print(f"python {platform.python_version()}  |  torch {torch.__version__}")
print(f"cpu cores: {os.cpu_count()}")
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print(f"gpu: {properties.name}  |  {properties.total_memory / 1024**3:.1f} GiB")
    # bf16 needs capability >= 8.0. A T4 is 7.5, so AMP runs fp16 + GradScaler.
    print(f"capability {properties.major}.{properties.minor}  bf16={properties.major >= 8}")
else:
    print("NO GPU. Runtime > Change runtime type > GPU, then rerun.")

## 2. Setup

Idempotent: safe to re-run at any point, and every step short-circuits if it is already
done. It executes on the **Colab VM**, not your laptop — the notebook file can live locally,
but its cells only ever see the VM filesystem.

Three things here are workarounds for real failures, not ceremony:

- **`--ignore-requires-python`.** Colab is on Python 3.13; `pyproject.toml` pins
  `>=3.12,<3.13`, so pip refuses the install outright. The override is local and changes
  nothing in the repo. (The proper fix is to widen that pin.)
- **`sys.path` insertion.** An editable install drops a `.pth` into site-packages, but a
  kernel started *before* the install never reads it. Without this the import fails even
  though pip reported success.
- **`os.chdir(REPO)`.** Config paths like `data/splits` are relative, and the kernel starts
  in `/content`.

In [ ]:
import shutil, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


def find_repository_root() -> Path | None:
    # Upward from cwd covers running inside the repo; the extra candidates cover
    # Colab, where the kernel starts in /content and the clone sits *below* it.
    start = Path.cwd()
    for candidate in [start, *start.parents, Path("/content/fdl-project"),
                      Path.home() / "fdl-project"]:
        if looks_like_the_repository(candidate):
            return candidate
    return None


REPO = find_repository_root()
if REPO is None:
    import getpass

    target = Path("/content/fdl-project")
    token = getpass.getpass("GitHub token (hidden, not saved): ")
    assert os.system(f"git clone --quiet https://{token}@github.com/ezero3/fdl-project.git {target}") == 0
    del token
    REPO = target

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyyaml", "scikit-learn", "matplotlib"], check=True)

source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import importlib
importlib.invalidate_caches()
import fdl_project

print(f"repository {REPO}")
print(f"package    {Path(fdl_project.__file__).parent}")

### Dataset

`WM811K.pkl` is gitignored (~1.9 GiB), so it is never in a fresh clone. Three routes,
cheapest first — each a no-op if an earlier one already succeeded.

**Take it from the original source, not a Kaggle re-upload.** `data/splits/` holds **row
indices into the pickle**, not copies of the maps, so they only mean anything against the
exact serialization they were built from. A different upload with a different row order
would silently select different wafers, break the lot-grouping, and open the door to
train/test leakage with nothing failing loudly. Hence the exact byte check.

**Train from `/content`, never from Drive directly** — Drive is a FUSE network mount, fine
for one sequential read and poor for the repeated access a run makes.

In [ ]:
import numpy as np

DATASET = REPO / "data" / "MIR-WM811K" / "WM811K.pkl"
SPLITS = REPO / "data" / "splits"
SOURCE_URL = "http://mirlab.org/dataset/public/MIR-WM811K.zip"
DRIVE_DATASET = Path("/content/drive/MyDrive/BICOCCA/FDL/DATA/data/MIR-WM811K/WM811K.pkl")
EXPECTED_BYTES = 2_022_961_642
TRAIN_ROWS, VALIDATION_ROWS = 121_063, 34_591


def ensure_drive() -> Path | None:
    # Returns immediately if already mounted, so re-running never re-prompts.
    # `force_remount` is deliberately unused -- that is what tears down a live mount.
    root = Path("/content/drive/MyDrive")
    if root.is_dir():
        return root
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return None
    return root if root.is_dir() else None


def fetch_from_source(destination: Path) -> None:
    import tempfile, urllib.request, zipfile

    with tempfile.TemporaryDirectory() as work:
        archive = Path(work) / "MIR-WM811K.zip"
        print(f"  downloading (~328 MB) from {SOURCE_URL}")
        urllib.request.urlretrieve(SOURCE_URL, archive)
        # The archive also carries a 3.6 GB MATLAB .mat; stream out just the
        # pickle rather than extracting 5.6 GB to reach it.
        with zipfile.ZipFile(archive) as bundle:
            member = next(n for n in bundle.namelist() if n.endswith("WM811K.pkl"))
            with bundle.open(member) as src, open(destination, "wb") as dst:
                shutil.copyfileobj(src, dst)


DATASET.parent.mkdir(parents=True, exist_ok=True)
if DATASET.exists():
    route = "already on this VM"
else:
    drive_root = ensure_drive()
    if drive_root is not None and DRIVE_DATASET.exists():
        print("  copying from Drive")
        shutil.copy2(DRIVE_DATASET, DATASET)
        route = "copied from Drive"
    else:
        fetch_from_source(DATASET)
        route = "downloaded"
        if drive_root is not None:
            DRIVE_DATASET.parent.mkdir(parents=True, exist_ok=True)
            print("  caching to Drive for next session")
            shutil.copy2(DATASET, DRIVE_DATASET)
            route = "downloaded, cached to Drive"

size = DATASET.stat().st_size
if size != EXPECTED_BYTES:
    raise SystemExit(f"unexpected size {size:,} (want {EXPECTED_BYTES:,}) -- wrong upload?")

print(f"dataset  {size / 1024**3:.2f} GiB  [{route}]")
print(f"drive    {'mounted' if Path('/content/drive/MyDrive').is_dir() else 'not mounted'}")
for split in ("train", "validation", "test"):
    path = SPLITS / f"{split}_indices.npy"
    print(f"  {split:<11}{len(np.load(path)):>8,} rows" if path.exists() else f"  {split}: MISSING")

## 3. Timing harness

Calls the **real** `train_one_epoch` and `evaluate_model` the runner uses, over a stratified
subset, then extrapolates to the full split. Measuring a subset is what keeps the sweeps to
minutes; it is sound because throughput is flat across an epoch.

- **The first pass is discarded** — cuDNN autotuning, worker startup and CUDA context
  creation all land in it and none recur. Every model therefore takes about twice its
  reported time to measure.
- **GPU utilisation is sampled** from `nvidia-smi`, not inferred. It is the number that
  decides whether batch size matters at all: at 15% the dataloader is the limit and a bigger
  batch cannot help; at 98% the card is the limit and it can.

In [ ]:
import logging, threading, time
from dataclasses import asdict, dataclass

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.data.imbalance import build_training_loss, compute_class_counts
from fdl_project.evaluation import evaluate_model
from fdl_project.models.baseline_cnn import count_trainable_parameters
from fdl_project.training.loop import resolve_device, train_one_epoch
from fdl_project.training.optim import build_optimizer
from fdl_project.training.runner import build_dataloaders, build_datasets
from fdl_project.training.seed import seed_everything

logging.basicConfig(level=logging.WARNING)
CONFIGS = REPO / "configs" / "train"


class GpuSampler:
    def __init__(self, interval: float = 0.2) -> None:
        self.interval, self.samples, self._stop = interval, [], threading.Event()

    def _poll(self) -> None:
        command = ["nvidia-smi", "--query-gpu=utilization.gpu",
                   "--format=csv,noheader,nounits"]
        while not self._stop.is_set():
            try:
                out = subprocess.run(command, capture_output=True, text=True, timeout=5).stdout
                self.samples.append(float(out.strip().split("\n")[0]))
            except Exception:
                pass
            self._stop.wait(self.interval)

    def __enter__(self):
        if torch.cuda.is_available():
            threading.Thread(target=self._poll, daemon=True).start()
        return self

    def __exit__(self, *exception):
        self._stop.set()

    @property
    def mean_utilisation(self) -> float:
        return float(np.mean(self.samples)) if self.samples else float("nan")


@dataclass
class Timing:
    label: str
    model: str
    batch_size: int
    parameters: int
    train_samples_per_second: float
    validation_samples_per_second: float
    train_epoch_seconds: float
    validation_epoch_seconds: float
    full_epoch_seconds: float
    peak_memory_gib: float
    gpu_utilisation: float
    fits: bool = True
    note: str = ""

In [ ]:
#: Rows per split for one timing pass. Large enough to amortise startup, small
#: enough that the sweeps take minutes. Drop it for 224x224 work.
TIMING_SUBSET = 20_000


def measure(config_path, *, label, overrides=None, dataframe=None, subset=None) -> Timing:
    overrides = [f"data.subset={subset or TIMING_SUBSET}", *(overrides or [])]
    config = load_experiment_config(config_path, overrides=overrides)
    seed_everything(config.seed)
    device = resolve_device(config.trainer.device)

    train_dataset, validation_dataset = build_datasets(config, dataframe)
    train_loader, validation_loader = build_dataloaders(config, train_dataset, validation_dataset)
    criterion = build_training_loss(
        config.imbalance, compute_class_counts(train_dataset.target_indices)
    )
    model = build_model(config.model.name, **dict(config.model.kwargs)).to(device)
    optimizer = build_optimizer(model, config.optimizer)
    scaler = torch.amp.GradScaler(
        device.type, enabled=config.trainer.amp and device.type == "cuda"
    )

    def one_training_pass() -> float:
        started = time.monotonic()
        train_one_epoch(model, train_loader, optimizer, criterion, device=device,
                        max_gradient_norm=config.trainer.max_gradient_norm, scaler=scaler)
        if device.type == "cuda":
            torch.cuda.synchronize()
        return time.monotonic() - started

    try:
        one_training_pass()  # discarded warmup
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        with GpuSampler() as sampler:
            train_seconds = one_training_pass()
            started = time.monotonic()
            evaluate_model(model, validation_loader, device=device, split_name="validation")
            if device.type == "cuda":
                torch.cuda.synchronize()
            validation_seconds = time.monotonic() - started
    except torch.cuda.OutOfMemoryError as error:
        torch.cuda.empty_cache()
        return Timing(label, config.model.name, config.trainer.batch_size,
                      count_trainable_parameters(model), *([float("nan")] * 7),
                      fits=False, note=str(error).split("\n")[0][:90])

    peak = torch.cuda.max_memory_allocated() / 1024**3 if device.type == "cuda" else 0.0
    train_rate = len(train_dataset) / train_seconds
    validation_rate = len(validation_dataset) / validation_seconds
    train_epoch, validation_epoch = TRAIN_ROWS / train_rate, VALIDATION_ROWS / validation_rate

    timing = Timing(label, config.model.name, config.trainer.batch_size,
                    count_trainable_parameters(model), train_rate, validation_rate,
                    train_epoch, validation_epoch, train_epoch + validation_epoch,
                    peak, sampler.mean_utilisation)
    del model, optimizer, train_loader, validation_loader, train_dataset, validation_dataset
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return timing


def report(rows) -> pd.DataFrame:
    frame = pd.DataFrame([asdict(r) for r in rows])
    columns = ["label", "batch_size", "fits", "train_epoch_seconds",
               "full_epoch_seconds", "peak_memory_gib", "gpu_utilisation"]
    print(frame[columns].round(2).to_string(index=False))
    return frame


started = time.monotonic()
dataframe = load_wm811k_dataframe(DATASET)
print(f"{len(dataframe):,} rows loaded in {time.monotonic() - started:.0f}s")

## 4. The three phase-1 candidates

Stock pipeline, batch 512, 64x64 — the cheap from-scratch models the plan proposes as the
test-bed for the whole phase-2 sweep.

Measured: **23.6s / 168.4s / 163.6s**. The 7x gap is the whole reason `baseline_cnn` is the
test-bed: it gets run ~16 more times, and the phase-1 tie-break on cost is decided by a
factor of seven rather than a rounding error.

In [ ]:
CANDIDATES = ["baseline_cnn_64", "dilated_style_64", "densenet_style_64"]
candidates = [measure(CONFIGS / f"{name}.yaml", label=name, dataframe=dataframe)
              for name in CANDIDATES]
table = report(candidates)

## 5. Batch size

Throughput rises with batch size until the GPU saturates, then flattens. The rule is **pick
the smallest batch still on the flat part**, not the largest that fits — past saturation a
bigger batch buys no speed and costs optimizer steps.

Measured on `baseline_cnn`: **completely flat** (5,642 -> 6,403 samples/s for a 16x batch
increase) at 14-20% GPU utilisation. So batch size was never the lever here, and 512 was
close to the worst available choice: it gives 237 steps per epoch against 946 at 128, for
the same wall-clock.

"Just fill the VRAM" is right when the model is big enough that filling memory and
saturating compute coincide — `vit_b_16` at 224x224, below. It is wrong for a
157k-parameter CNN at 64x64, where the card is 80% idle and the CPU is the limit.

In [ ]:
TESTBED = "baseline_cnn_64"
sweep = [measure(CONFIGS / f"{TESTBED}.yaml", label=f"batch {size}",
                 overrides=[f"trainer.batch_size={size}"], dataframe=dataframe)
         for size in (64, 128, 256, 512, 1024)]

curve = pd.DataFrame([asdict(r) for r in sweep])
curve["steps_per_epoch"] = np.ceil(TRAIN_ROWS / curve["batch_size"]).astype(int)
curve["updates_at_20_epochs"] = curve["steps_per_epoch"] * 20
print(curve[["batch_size", "train_samples_per_second", "train_epoch_seconds",
             "steps_per_epoch", "updates_at_20_epochs", "peak_memory_gib",
             "gpu_utilisation"]].round(2).to_string(index=False))

## 6. Dataloader width — the actual lever

Flat throughput at 15% GPU utilisation says the bottleneck is upstream of the GPU: the
letterbox and one-hot encoding run per batch on CPU, by design, because augmentation makes
the transform output differ every epoch anyway.

**Measured 1.9x from one config line.** `defaults.yaml` shipped `num_workers: 2` on a
machine with 8 cores. Even at 8 workers utilisation only reaches ~22%, so this is still
CPU-bound — there is more left here, but not from the GPU side.

In [ ]:
workers = [measure(CONFIGS / f"{TESTBED}.yaml", label=f"workers {n}",
                   overrides=["trainer.batch_size=256", f"data.num_workers={n}"],
                   dataframe=dataframe)
           for n in (2, 4, 8)]
worker_table = report(workers)

## 7. The 224x224 pretrained arm

Where the budget actually goes. These run at 95-99% GPU utilisation — genuinely
compute-bound, the opposite regime to the 64x64 models, and the only place a faster card
would pay.

Measured, with dihedral-8 augmentation on:

| arm | best batch | ceiling | epoch | 25-epoch run |
|---|---|---|---|---|
| `vit_b_16` frozen encoder | 128 (1.35 GiB) | 1024 | 766s | 5.3 h |
| `vit_b_16` full fine-tune | 128 (9.80 GiB) | 128 | 1482s | 10.3 h |
| `vit_b_32` full fine-tune | 128 (3.47 GiB) | >256 | 458s | 3.2 h |

**Freezing only buys 1.9x, not the 10x people expect.** Removing the encoder backward halves
the cost; the 86M-parameter forward pass still runs on every sample. A "cheap frozen
baseline" is 5.3 hours, so budget it as a real run.

`vit_b_32` is the cheap transformer counterpart — same 86M parameters but 32x32 patches, so
49 tokens instead of 196 and ~4x less attention compute. There is no `vit_small` in
torchvision.

In [ ]:
# 224x224 transformers are slow; a smaller subset keeps this to minutes.
VIT_SUBSET = 4_000
vit_arms = []
for label, extra in [
    ("frozen", ["model.kwargs.freeze_encoder=true", "optimizer.param_groups=[]"]),
    ("finetune", []),
    ("vit_b_32", ["model.name=vit_b_32"]),
]:
    for size in (128, 256):
        vit_arms.append(measure(
            CONFIGS / "vit_b_16_224_finetune.yaml", label=f"{label} b{size}",
            overrides=[f"trainer.batch_size={size}", "data.augmentation.name=dihedral8",
                       *extra],
            dataframe=dataframe, subset=VIT_SUBSET))
vit_table = report(vit_arms)

## 8. Budget, and whether cross-validation fits

Phases 1 and 3 are where folds change a decision; phase 2's ~16 sweep arms stay on a single
split because K x 16 is the expensive one.

Bootstrap resamples the same wafers; cross-validation retrains on different ones, so the
across-fold spread is the better basis for choosing between models whose intervals overlap.

In [ ]:
EXPECTED_EPOCHS = 25   # early stopping (patience 5) usually fires well before 40
EPOCH_SECONDS = dict(zip(CANDIDATES, table["full_epoch_seconds"]))
mean_epoch = float(np.mean(list(EPOCH_SECONDS.values())))

rows = []
for testbed, seconds in EPOCH_SECONDS.items():
    for folds in (1, 3, 5):
        sweep_hours = 16 * EXPECTED_EPOCHS * seconds / 3600
        folded = (3 + 10) * folds * EXPECTED_EPOCHS * mean_epoch / 3600
        rows.append({"test-bed": testbed, "K": folds,
                     "phase 2 (h)": round(sweep_hours, 1),
                     "phases 1+3 (h)": round(folded, 1),
                     "total (h)": round(sweep_hours + folded, 1),
                     "each of 3 (h)": round((sweep_hours + folded) / 3, 1)})
budget = pd.DataFrame(rows)
print(budget.to_string(index=False))
print("\n64x64 only -- the 224x224 arm above is extra and dominates it.")

In [ ]:
import json
from datetime import datetime, timezone

OUTPUT = REPO / "output" / "phase0"
OUTPUT.mkdir(parents=True, exist_ok=True)
for name, frame in [("model_timings", table), ("batch_size_curve", curve),
                    ("num_workers", worker_table), ("vit_224_timings", vit_table),
                    ("budget", budget)]:
    frame.to_csv(OUTPUT / f"{name}.csv", index=False)
(OUTPUT / "environment.json").write_text(json.dumps({
    "recorded": datetime.now(timezone.utc).isoformat(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
    if torch.cuda.is_available() else None,
    "cpu_cores": os.cpu_count(), "torch": torch.__version__,
    "python": platform.python_version(), "timing_subset": TIMING_SUBSET,
}, indent=2))

# Colab local disk dies with the session; Drive does not.
if Path("/content/drive/MyDrive").is_dir():
    archive = Path("/content/drive/MyDrive/BICOCCA/FDL/phase0")
    archive.mkdir(parents=True, exist_ok=True)
    for path in OUTPUT.iterdir():
        shutil.copy2(path, archive / path.name)
    print("copied to", archive)
print("\n".join(f"  {p.name}" for p in sorted(OUTPUT.iterdir())))

## 9. Settings these measurements imply

Applied to `configs/train/defaults.yaml` unless noted.

```yaml
data:
  num_workers: 8          # was 2; measured 1.9x, free
trainer:
  batch_size: 256         # was 512; same speed, 2x the optimizer steps
```

Per-config, for the 224x224 arm:

```yaml
# vit_b_16_224_finetune.yaml
trainer:
  batch_size: 128         # was 32; fits in 9.8 GiB and is 7% faster
```

**If you change batch size, rescale the learning rate** — square-root scaling for
Adam-family optimizers, so 512 -> 256 takes `lr` 1.0e-3 -> ~7.0e-4. A batch-size comparison
at fixed learning rate is not a batch-size comparison.

**Then hold both fixed.** Batch size and worker count are part of the pipeline: phase 2's
arms are only controlled if every arm shares them. Phase 3 is the exception — batch size,
learning rate, input size and epoch budget are the four things that do not transfer across
architectures and must be re-tuned per model.

### Which GPU

| workload | measured utilisation | card |
|---|---|---|
| 64x64 `baseline_cnn` | 15.7% (2w) / 22% (8w) | **T4** — dataloader-bound; a faster card idles more |
| 64x64 `dilated`/`densenet` | 96-97% | **L4** — compute-bound, and roughly cost-neutral |
| 224x224 pretrained | 95-99% | **A100** — the budget sink, and the one place to spend |

Compute units burn at roughly T4 1x, L4 ~2.5x, A100 ~6x per hour, against speedups of ~2x
and ~4-5x. So L4 is close to cost-neutral and A100 is deliberately buying wall-clock with
budget — worth doing once, on the 224 arm, not on 64x64 screening that was already cheap.

Speedup figures are spec-ratio estimates; only the T4 column is measured.